# Torlak Morphological Feature: **Definite** (Fixed)
## XLM-RoBERTa-large · Feature-pair probing

**Fix:** The original notebook produced the merged class `Art,Def`.
This notebook normalises multi-value `Definite` annotations so that
`Art,Def` → `Art`, giving four clean, separate classes:

| Class | Meaning |
|---|---|
| `_` | Feature absent on this token |
| `Art` | Definite article form (postpositive article in Torlak) |
| `Def` | Grammatically definite (referent is identifiable) |
| `Ind` | Indefinite |

Everything else is identical to the main `TorlakTag_FeatDifficulty_XLMRoberta2` notebook.

## 0 · Setup

In [ ]:
# =========================
# 0) SETUP
# =========================
!pip -q install lxml

import os, re, json, math, time, random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup

from google.colab import drive
drive.mount("/content/drive")

print("🧠 torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1 · Configuration

In [ ]:
# =========================
# 1) CONFIG
# =========================

DATA_ROOT     = Path("/content/drive/MyDrive/TorlakData")
MTE2UD_PATH   = DATA_ROOT / "mte2ud_output.txt"
SPK_META_PATH = DATA_ROOT / "spk.metadata.txt"
EXB_DIR       = DATA_ROOT / "exb_corrected"
GEO_TSV       = None

# Save this run's artefacts alongside the main experiment
MODELS_ROOT = Path("/content/drive/MyDrive/TorlakTag/feat_difficulty")
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

OUT_DIR = MODELS_ROOT / "definite_fixed"   # separate folder so the original is not overwritten
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME    = "FacebookAI/xlm-roberta-large"
EPOCHS        = 7
WARMUP_RATIO  = 0.06
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
PATIENCE      = 4
FREEZE_EPOCHS = 1

BS       = 24
ACCUM    = 4      # effective batch = 96
LR       = 1e-5
MAX_LEN  = 16
USE_FP16 = True

MIN_FEAT_VAL_FREQ      = 5
MIN_FEAT_TYPE_EXAMPLES = 100

SEED = 13
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("📁 OUT_DIR:", OUT_DIR)
print("🧠 device:", device)

## 2 · Token normalisation

In [ ]:
# =========================
# 2) TOKEN NORMALIZATION (verbatim from original)
# =========================

def apply_spec_mapping(s: str) -> str:
    if s is None:
        return ""
    s = s.replace("#", "")
    s = s.replace("W", "Ə").replace("w", "ə")
    s = s.replace("1", "ḱ").replace("6", "ḱ")
    s = s.replace("2", "ǵ")
    s = s.replace("3", "č")
    s = s.replace("x", "š").replace("X", "š")
    s = s.replace("5", "ƨ")
    s = s.replace("ššš", "XXX")
    return s

RE_OVERLAP   = re.compile(r"\[[^\]]*\]")
RE_DOUBLEPAR = re.compile(r"^\(\(.*\)\)$")
RE_BULLETS   = re.compile(r"^[•]+$")
RE_LONGVOWEL = re.compile(r"([aeiouə])\1+")
RE_SPACES    = re.compile(r"\s+")

_STRIP_EDGE = " \t\r\n\"'\u201c\u201d\u201e`´.,;:!?(){}[]<>•"

def is_x_special_token(tok: str) -> bool:
    t = tok.strip()
    if not t:
        return True
    if RE_DOUBLEPAR.match(t):
        return True
    if RE_BULLETS.match(t):
        return True
    return False

def strip_attached_specials(tok: str) -> str:
    t = tok.strip()
    t = re.sub(r"/+$", "", t)
    t = t.strip(_STRIP_EDGE)
    return t

def normalize_word(tok: str) -> str:
    t = apply_spec_mapping(tok).lower()
    t = RE_LONGVOWEL.sub(r"\1", t)
    t = RE_SPACES.sub(" ", t).strip()
    return t

def tokenize_with_rules(raw_text: str):
    if raw_text is None:
        return [], []
    s = apply_spec_mapping(raw_text).lower()
    s = RE_OVERLAP.sub(" ", s)

    raw_tokens = [t for t in s.split() if t.strip()]
    tokens, special = [], []
    for rt in raw_tokens:
        if is_x_special_token(rt):
            tokens.append(rt)
            special.append(True)
            continue
        has_alnum = any(ch.isalpha() or ch.isdigit() for ch in rt)
        if has_alnum:
            w = strip_attached_specials(rt)
            w = normalize_word(w)
            if w:
                tokens.append(w)
                special.append(False)
            else:
                tokens.append(rt)
                special.append(True)
        else:
            tokens.append(rt)
            special.append(True)
    return tokens, special

print("✅ normalisation helpers ready")

## 3 · Load MTE→UD mapping + data splits

In [ ]:
# =========================
# 3) LOAD MTE→UD MAPPING + SPLITS (verbatim from original)
# =========================

UPOS_SET = {
    "ADJ","ADP","ADV","AUX","CCONJ","DET","INTJ","NOUN","NUM","PART",
    "PRON","PROPN","PUNCT","SCONJ","SYM","VERB","X"
}

def load_mte2ud(path: Path):
    m = {}
    bad = 0
    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = re.split(r"\t+", line)
            if len(parts) < 3:
                parts = re.split(r"\s{2,}", line)
            if len(parts) < 3:
                bad += 1
                continue
            mte  = parts[0].strip()
            upos = parts[1].strip()
            if len(parts) >= 4 and parts[2].strip() in UPOS_SET and upos in UPOS_SET:
                feats = parts[3].strip()
            else:
                feats = parts[2].strip()
            feats = feats if feats else "_"
            m[mte] = (upos if upos else "X", feats)
    print(f"✅ loaded MTE→UD mapping: {len(m)} tags (skipped {bad} broken lines)")
    return m

mte2ud = load_mte2ud(MTE2UD_PATH)

def find_split_file(root: Path, stem: str) -> Path:
    for ext in ["", ".txt", ".tsv", ".conllu", ".conll", ".data"]:
        p = root / f"{stem}{ext}"
        if p.exists():
            return p
    hits = [h for h in root.rglob(f"{stem}*") if h.is_file()]
    if hits:
        return sorted(hits, key=lambda x: len(str(x)))[0]
    raise FileNotFoundError(f"Could not find split file for '{stem}' under {root}")

TRAIN_PATH = find_split_file(DATA_ROOT, "tor_train")
DEV_PATH   = find_split_file(DATA_ROOT, "tor_dev")
TEST_PATH  = find_split_file(DATA_ROOT, "tor_test")
print("📄 train:", TRAIN_PATH)
print("📄 dev  :", DEV_PATH)
print("📄 test :", TEST_PATH)

def read_tok_lemma_mte(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) < 3:
                parts = line.split()
            if len(parts) < 3:
                continue
            form, lemma, mte = parts[0], parts[1], parts[2]
            tks, sp = tokenize_with_rules(form)
            if not tks:
                continue
            form_n   = tks[0]
            is_special = sp[0]
            if is_special:
                rows.append((form_n, form_n, "X"))
                continue
            lemma_n = normalize_word(strip_attached_specials(lemma))
            if not lemma_n:
                lemma_n = form_n
            rows.append((form_n, lemma_n, mte))
    return rows

train_raw = read_tok_lemma_mte(TRAIN_PATH)
dev_raw   = read_tok_lemma_mte(DEV_PATH)
test_raw  = read_tok_lemma_mte(TEST_PATH)
print(f"✅ loaded splits: train={len(train_raw)} dev={len(dev_raw)} test={len(test_raw)}")

def to_ud_example(row):
    form, lemma, xpos = row
    if xpos == "X":
        return (form, lemma, "X", "_", "X")
    upos, feats = mte2ud.get(xpos, ("X", "_"))
    return (form, lemma, upos, feats, xpos)

train_ex = [to_ud_example(r) for r in train_raw]
dev_ex   = [to_ud_example(r) for r in dev_raw]
test_ex  = [to_ud_example(r) for r in test_raw]
print("Converted sample:", train_ex[:5])

## 4 · Label maps

In [ ]:
# =========================
# 4) LABEL MAPS (verbatim from original)
# =========================

def build_vocab(items, min_freq=1, specials=None):
    specials = specials or []
    c = Counter(items)
    vocab = {}
    for sp in specials:
        vocab[sp] = len(vocab)
    for k, v in c.most_common():
        if k in vocab:
            continue
        if v >= min_freq:
            vocab[k] = len(vocab)
    return vocab

def ensure_special(v: Dict[str, int], key: str):
    if key not in v:
        v[key] = len(v)
    return v

upos_items = [u for _, _, u, _, _ in train_ex]
upos2id    = build_vocab(upos_items, min_freq=1, specials=["X", "_"])
upos2id    = ensure_special(upos2id, "X")
upos2id    = ensure_special(upos2id, "_")
id2upos    = {i: s for s, i in upos2id.items()}

print(f"✅ UPOS vocab: {len(upos2id)} classes → {list(upos2id.keys())}")

## 5 · Definite feature normalisation

**The fix lives here.**

The MTE→UD mapping produces `Definite=Art,Def` for postpositive-article forms.
In UD this is a valid multi-valued annotation, but for a single-label classifier
it conflates two distinct classes.  We normalise with a priority rule:

```
Art,Def  →  Art   (article function is the morphologically distinctive one)
```

The resulting label set is `{_, Art, Def, Ind}`.

In [ ]:
# =========================
# 5) DEFINITE FEATURE — NORMALISED
# =========================

def parse_feats(feats_str: str) -> Dict[str, str]:
    """'Case=Nom|Gender=Masc|Number=Sing' → {'Case': 'Nom', ...}"""
    if not feats_str or feats_str == "_":
        return {}
    result = {}
    for kv in feats_str.split("|"):
        if "=" in kv:
            k, v = kv.split("=", 1)
            result[k.strip()] = v.strip()
    return result


# Priority order for multi-valued Definite annotations:
#   'Art,Def' → 'Art'  (first value wins; article is the primary function)
_DEFINITE_NORMALISE = {
    "Art,Def": "Art",
    "Def,Art": "Art",   # guard against reversed ordering
}

def get_definite_value(feats_str: str) -> str:
    """Return the normalised Definite value, or '_' if the feature is absent."""
    raw = parse_feats(feats_str).get("Definite", "_")
    return _DEFINITE_NORMALISE.get(raw, raw)


# --- Frequency audit before/after normalisation ---
raw_counter  = Counter()
norm_counter = Counter()

for _, _, _, feats, _ in train_ex:
    raw_val  = parse_feats(feats).get("Definite", "_")
    norm_val = get_definite_value(feats)
    raw_counter[raw_val]  += 1
    norm_counter[norm_val] += 1

print("Raw  Definite value counts (train):")
for v, c in raw_counter.most_common():
    print(f"  {v!r:<15} {c}")

print("\nNormalised Definite value counts (train):")
for v, c in norm_counter.most_common():
    print(f"  {v!r:<15} {c}")

## 6 · Dataset, model, helpers

In [ ]:
# =========================
# 6) DATASET / MODEL
# =========================

class DefiniteDataset(Dataset):
    """
    Same as FeatPairDataset from the original notebook, but uses
    get_definite_value() for label lookup instead of generic parse_feats().
    """
    def __init__(self, examples, feat2id: Dict[str, int]):
        self.items = []
        for form, _lemma, upos, feats, _xpos in examples:
            up_id = upos2id.get(upos, upos2id["X"])
            fval  = get_definite_value(feats)
            fe_id = feat2id.get(fval, feat2id.get("<UNK>", feat2id["_"]))
            self.items.append((form, up_id, fe_id))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def make_collate(tokenizer, max_length: int):
    def collate(items):
        tokens = [it[0] for it in items]
        up_y   = torch.tensor([it[1] for it in items], dtype=torch.long)
        fe_y   = torch.tensor([it[2] for it in items], dtype=torch.long)
        enc = tokenizer(tokens, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"], up_y, fe_y
    return collate


def get_hidden_size_from_config(cfg):
    if hasattr(cfg, "hidden_size"):
        return int(cfg.hidden_size)
    if hasattr(cfg, "d_model"):
        return int(cfg.d_model)
    raise ValueError("Cannot infer hidden size from config.")


def mean_pool(last_hidden, attention_mask):
    mask   = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom  = mask.sum(dim=1).clamp(min=1.0)
    return summed / denom


class TwoHeadTagger(nn.Module):
    """XLM-RoBERTa-large with UPOS head + Definite head."""
    def __init__(self, encoder_name: str, n_upos: int, n_feat: int, dropout: float = 0.1):
        super().__init__()
        cfg = AutoConfig.from_pretrained(encoder_name)
        self.encoder   = AutoModel.from_pretrained(encoder_name, torch_dtype=torch.float32)
        h              = get_hidden_size_from_config(cfg)
        self.drop      = nn.Dropout(dropout)
        self.upos_head = nn.Linear(h, n_upos)
        self.feat_head = nn.Linear(h, n_feat)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        x   = self.drop(mean_pool(out.last_hidden_state, attention_mask))
        return self.upos_head(x), self.feat_head(x)


@torch.no_grad()
def evaluate(model, loader, feat2id: Dict[str, int], amp_dtype=None):
    model.eval()
    total = corr_up = corr_fe = corr_fe_present = n_present = 0
    absent_id = feat2id["_"]

    for input_ids, attn, up_y, fe_y in loader:
        input_ids, attn = input_ids.to(device), attn.to(device)
        up_y, fe_y      = up_y.to(device), fe_y.to(device)

        if amp_dtype is not None and device.type == "cuda":
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                up_l, fe_l = model(input_ids, attn)
        else:
            up_l, fe_l = model(input_ids, attn)

        up_p = up_l.argmax(dim=1)
        fe_p = fe_l.argmax(dim=1)

        corr_up += int((up_p == up_y).sum().item())
        corr_fe += int((fe_p == fe_y).sum().item())

        present_mask     = fe_y != absent_id
        n_present       += int(present_mask.sum().item())
        corr_fe_present += int(((fe_p == fe_y) & present_mask).sum().item())
        total           += input_ids.size(0)

    return {
        "upos_acc":         corr_up / max(1, total),
        "feat_acc":         corr_fe / max(1, total),
        "feat_acc_present": corr_fe_present / max(1, n_present),
        "n":         total,
        "n_present": n_present,
    }


print("✅ model / dataset helpers ready")

## 7 · Build feat2id for Definite (normalised)

In [ ]:
# =========================
# 7) BUILD feat2id — SEPARATE Art / Def / Ind CLASSES
# =========================

# Count normalised values in the training set
definite_val_counts = Counter()
for _, _, _, feats, _ in train_ex:
    v = get_definite_value(feats)
    if v != "_":    # count only present tokens
        definite_val_counts[v] += 1

print("Normalised Definite values (train, present tokens):")
for v, c in definite_val_counts.most_common():
    print(f"  {v!r:<15} {c}")

# Build label vocab
#   id=0 → '_'  (absent class — consistent with original)
feat2id = {"_": 0}
for val, cnt in definite_val_counts.most_common():
    if cnt >= MIN_FEAT_VAL_FREQ:
        feat2id[val] = len(feat2id)
feat2id["<UNK>"] = len(feat2id)

id2feat = {i: v for v, i in feat2id.items()}

print(f"\nfeat2id ({len(feat2id)} classes): {feat2id}")
print("Expected classes: '_', 'Art', 'Def', 'Ind', '<UNK>'")

## 8 · Training loop

In [ ]:
# =========================
# 8) TRAIN — Definite (fixed)
# =========================

FEAT_TYPE = "Definite"

# Save vocabs
json.dump(feat2id, open(OUT_DIR / "feat2id.json",  "w", encoding="utf-8"), ensure_ascii=False, indent=2)
json.dump(upos2id, open(OUT_DIR / "upos2id.json",  "w", encoding="utf-8"), ensure_ascii=False, indent=2)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.save_pretrained(str(OUT_DIR))

collate      = make_collate(tokenizer, MAX_LEN)
train_ds     = DefiniteDataset(train_ex, feat2id)
dev_ds       = DefiniteDataset(dev_ex,   feat2id)
test_ds      = DefiniteDataset(test_ex,  feat2id)

train_loader = DataLoader(train_ds, batch_size=BS,    shuffle=True,  num_workers=2,
                          collate_fn=collate, pin_memory=True)
dev_loader   = DataLoader(dev_ds,   batch_size=BS*2,  shuffle=False, num_workers=2,
                          collate_fn=collate, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS*2,  shuffle=False, num_workers=2,
                          collate_fn=collate, pin_memory=True)

model = TwoHeadTagger(MODEL_NAME, len(upos2id), len(feat2id), dropout=0.1).to(device)

# Freeze encoder for first epoch
for p in model.encoder.parameters():
    p.requires_grad = False

optim = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY
)
total_steps  = EPOCHS * math.ceil(len(train_loader) / ACCUM)
warmup_steps = int(total_steps * WARMUP_RATIO)
sched = get_cosine_schedule_with_warmup(optim,
                                        num_warmup_steps=warmup_steps,
                                        num_training_steps=total_steps)

amp_dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else None
scaler    = torch.amp.GradScaler("cuda") if amp_dtype is not None else None

best_acc, best_epoch, bad = -1.0, -1, 0

def rebuild_optimizer_after_unfreeze():
    global optim, sched
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = get_cosine_schedule_with_warmup(optim,
                                            num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

log = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    loss_accum = 0.0
    seen = 0

    if epoch == FREEZE_EPOCHS + 1:
        for p in model.encoder.parameters():
            p.requires_grad = True
        rebuild_optimizer_after_unfreeze()
        print("🔥 Encoder unfrozen")

    optim.zero_grad(set_to_none=True)

    for step, (input_ids, attn, up_y, fe_y) in enumerate(train_loader, start=1):
        input_ids = input_ids.to(device, non_blocking=True)
        attn      = attn.to(device, non_blocking=True)
        up_y      = up_y.to(device, non_blocking=True)
        fe_y      = fe_y.to(device, non_blocking=True)

        if amp_dtype is not None:
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                up_l, fe_l = model(input_ids, attn)
                loss = (
                    nn.functional.cross_entropy(up_l, up_y) +
                    nn.functional.cross_entropy(fe_l, fe_y)
                ) / ACCUM
            scaler.scale(loss).backward()
        else:
            up_l, fe_l = model(input_ids, attn)
            loss = (
                nn.functional.cross_entropy(up_l, up_y) +
                nn.functional.cross_entropy(fe_l, fe_y)
            ) / ACCUM
            loss.backward()

        loss_accum += float(loss.item()) * input_ids.size(0) * ACCUM
        seen       += input_ids.size(0)

        if (step % ACCUM) == 0:
            if scaler is not None:
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optim)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optim.step()
            optim.zero_grad(set_to_none=True)
            sched.step()

    torch.save(model.state_dict(), OUT_DIR / "last_epoch.pt")

    dev_m      = evaluate(model, dev_loader, feat2id, amp_dtype=amp_dtype)
    train_loss = loss_accum / max(1, seen)
    dt         = time.time() - t0

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "dev_upos_acc":         dev_m["upos_acc"],
        "dev_feat_acc":         dev_m["feat_acc"],
        "dev_feat_acc_present": dev_m["feat_acc_present"],
        "minutes": dt / 60.0,
    }
    log.append(row)
    json.dump(log, open(OUT_DIR / "train_log.json", "w", encoding="utf-8"), indent=2)

    print(f"[{epoch:02d}] train_loss={train_loss:.4f}  "
          f"dev_upos={dev_m['upos_acc']*100:.2f}%  "
          f"dev_feat(present)={dev_m['feat_acc_present']*100:.2f}%  "
          f"n_present={dev_m['n_present']}  ({dt/60:.1f}m)")

    if dev_m["feat_acc_present"] > best_acc + 1e-6:
        best_acc, best_epoch, bad = dev_m["feat_acc_present"], epoch, 0
        torch.save(model.state_dict(), OUT_DIR / "best_model.pt")
        json.dump({"best_dev_feat_acc_present": best_acc, "best_epoch": best_epoch,
                   "feat_type": FEAT_TYPE, "model_name": MODEL_NAME,
                   "classes": list(feat2id.keys())},
                  open(OUT_DIR / "best_meta.json", "w", encoding="utf-8"), indent=2)
        print("💾 saved best_model.pt")
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"⏹️  Early stopping. best_dev_feat(present)={best_acc*100:.2f}% at epoch {best_epoch}")
            break

print("\n✅ Training complete.")

## 9 · Test evaluation

In [ ]:
# =========================
# 9) TEST — load best checkpoint
# =========================

best_model = TwoHeadTagger(MODEL_NAME, len(upos2id), len(feat2id), dropout=0.1).to(device)
best_model.load_state_dict(torch.load(OUT_DIR / "best_model.pt", map_location=device))
test_m = evaluate(best_model, test_loader, feat2id, amp_dtype=amp_dtype)

print(f"\n🧪 TEST RESULTS  (Definite — fixed, separate Art/Def classes)")
print(f"  UPOS acc         : {test_m['upos_acc']*100:.2f}%")
print(f"  Feat acc (all)   : {test_m['feat_acc']*100:.2f}%")
print(f"  Feat acc (present): {test_m['feat_acc_present']*100:.2f}%   ← main metric")
print(f"  n_present (test) : {test_m['n_present']}")
print(f"  Best epoch       : {best_epoch}")
print(f"  Classes          : {list(feat2id.keys())}")

test_m["feat_type"]   = FEAT_TYPE
test_m["best_epoch"]  = best_epoch
test_m["n_values"]    = len(feat2id) - 2
test_m["train_count"] = sum(1 for _, _, _, f, _ in train_ex if get_definite_value(f) != "_")
test_m["classes"]     = list(feat2id.keys())

json.dump(
    {k: float(v) if isinstance(v, (float, int)) else v for k, v in test_m.items()},
    open(OUT_DIR / "test_results.json", "w"), indent=2
)
print(f"\nSaved → {OUT_DIR / 'test_results.json'}")

# Also write this result into the shared all_results.json so the ranking table picks it up
results_json = MODELS_ROOT / "all_results.json"
all_results  = {}
if results_json.exists():
    all_results = json.load(open(results_json, encoding="utf-8"))

all_results["Definite"] = {k: float(v) if isinstance(v, (float, int)) else v for k, v in test_m.items()}
json.dump(all_results, open(results_json, "w", encoding="utf-8"), indent=2)
print(f"Updated → {results_json}  (Definite entry overwritten with fixed result)")

torch.cuda.empty_cache()

## 10 · Per-class breakdown

In [ ]:
# =========================
# 10) PER-CLASS ACCURACY
# =========================

best_model.eval()
absent_id = feat2id["_"]
class_correct = Counter()
class_total   = Counter()

with torch.no_grad():
    for input_ids, attn, up_y, fe_y in test_loader:
        input_ids, attn = input_ids.to(device), attn.to(device)
        fe_y            = fe_y.to(device)

        if amp_dtype is not None and device.type == "cuda":
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                _, fe_l = best_model(input_ids, attn)
        else:
            _, fe_l = best_model(input_ids, attn)

        fe_p = fe_l.argmax(dim=1)
        for true_id, pred_id in zip(fe_y.cpu().tolist(), fe_p.cpu().tolist()):
            if true_id == absent_id:
                continue   # skip absent tokens
            label = id2feat[true_id]
            class_total[label]   += 1
            class_correct[label] += int(true_id == pred_id)

print(f"{'Class':<12} {'Correct':>8} {'Total':>7} {'Acc (%)':>8}")
print("─" * 40)
for cls in [v for v in feat2id if v not in ("_", "<UNK>")]:
    tot = class_total[cls]
    if tot == 0:
        print(f"{cls:<12} {'—':>8} {'0':>7} {'—':>8}")
    else:
        acc = class_correct[cls] / tot * 100
        print(f"{cls:<12} {class_correct[cls]:>8} {tot:>7} {acc:>8.2f}")

## 11 · Visualisation

In [ ]:
# =========================
# 11) BAR CHART — per-class accuracy
# =========================

classes = [v for v in feat2id if v not in ("_", "<UNK>")]
accs    = [
    (class_correct[c] / class_total[c] * 100) if class_total[c] > 0 else 0.0
    for c in classes
]

colors = cm.RdYlGn(np.linspace(0.15, 0.85, len(classes)))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(classes, accs, color=colors)
for bar, val in zip(bars, accs):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=10)

ax.set_xlabel("Test accuracy on present tokens (%)", fontsize=11)
ax.set_title("Definite (fixed: Art / Def / Ind as separate classes)\n"
             "XLM-RoBERTa-large", fontsize=11)
ax.set_xlim(0, 110)
ax.axvline(x=50, color="gray", linestyle="--", alpha=0.4, label="50% baseline")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(str(OUT_DIR / "definite_fixed_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved →", OUT_DIR / "definite_fixed_accuracy.png")